# Evaluación Parcial 2: clasificación de hojas de tomate mediante una CNN

**Asignatura:** TLY1102 — Técnicas Avanzadas de Machine Learning I  
**Continuidad:** se reutiliza la misma variante y partición de la Evaluación Parcial 1.  
**Objetivo:** comparar experimentalmente una Red Neuronal Convolucional (CNN) contra el MLP anterior.

La CNN conserva la estructura matricial de la imagen `(64, 64, 3)` y aprende filtros locales mediante convoluciones. El MLP de la EP1 aplanaba la imagen con `Flatten`, perdiendo la relación espacial entre píxeles.

## Variante del grupo

- Cinco clases de tomate: `Tomato_healthy`, `Tomato_Early_blight`, `Tomato_Late_blight`, `Tomato_Leaf_Mold` y `Tomato_Bacterial_spot`.
- Máximo de 400 imágenes por clase: 2.000 imágenes.
- Semilla reproducible: `1102`.
- Partición estratificada: 70% entrenamiento, 15% validación y 15% prueba.
- Entrada de la CNN: imágenes RGB de 64 × 64 píxeles normalizadas a `[0, 1]`.

## Flujo de trabajo de la EP2

1. Mantener el problema, las clases y la partición de la EP1.
2. Preparar las imágenes como tensores, sin aplanarlas antes de la CNN.
3. Diseñar una arquitectura con convoluciones, pooling y capas densas.
4. Entrenar y evaluar la CNN con las mismas métricas del MLP.
5. Comparar Accuracy, Precision, Recall, F1, parámetros, tiempo y generalización.
6. Visualizar Feature Maps y mapas de activación por clase.
7. Analizar errores y justificar las diferencias entre ambas arquitecturas.

## 0. Configuración del entorno

Se reutiliza la lógica de carga y extracción selectiva del notebook de la EP1.

In [ ]:
# Solo se instala lo que falte (en Colab normalmente no falta nada).
# Si esta celda no imprime nada, el entorno ya estaba completo.
import importlib

for paquete in ["tensorflow", "sklearn", "seaborn", "pandas", "matplotlib", "PIL"]:
    if importlib.util.find_spec(paquete) is None:
        nombre_pip = {"sklearn": "scikit-learn", "PIL": "pillow"}.get(paquete, paquete)
        print(f"Instalando {nombre_pip}...")
        %pip install -q {nombre_pip}

In [ ]:
import os
import random
import time
import warnings
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    BatchNormalization,
    Activation,
    Flatten,
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 1102
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print("TensorFlow:", tf.__version__)
print("Semilla:", SEED)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

### Montaje de Google Drive

En Colab el dataset se lee desde Drive. **Antes de ejecutar**, sube
`PlantVillage.zip` (o la carpeta ya descomprimida) a tu unidad en:

```text
Mi unidad/PlantVillage_MLP/PlantVillage.zip
```

Si el notebook se ejecuta fuera de Colab, esta celda no hace nada y se usa la
carpeta local `data/`.

In [ ]:
IN_COLAB = importlib.util.find_spec("google.colab") is not None
print("Entorno:", "Google Colab" if IN_COLAB else "local")

DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/PlantVillage_MLP")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print("Carpeta de trabajo en Drive:", DRIVE_ROOT)

### Localización del dataset

Se busca el dataset en este orden:

1. Una carpeta ya descomprimida (en `/content` o en Drive, según el entorno).
2. Un `PlantVillage.zip` en Drive o en el proyecto → se descomprime solo las
   **cinco clases de tomate** que usa la variante, no las 15 del archivo completo.
3. La carpeta local `data/PlantVillage` (ejecución fuera de Colab).

Extraer solo lo necesario evita copiar ~40.000 archivos: son unos 7.500, y la
descompresión baja de varios minutos a segundos.

In [ ]:
import zipfile

IMAGE_EXTS = {".jpg", ".jpeg", ".png"}

# Las cinco clases de la variante. Se declara aquí porque la extracción
# selectiva del zip las necesita antes de la sección 2.
SELECTED_CLASSES = [
    "Tomato_healthy",
    "Tomato_Early_blight",
    "Tomato_Late_blight",
    "Tomato_Leaf_Mold",
    "Tomato_Bacterial_spot",
]


def es_raiz_valida(directorio):
    """True si el directorio contiene las cinco clases con imágenes dentro."""
    directorio = Path(directorio)

    if not directorio.is_dir():
        return False

    for clase in SELECTED_CLASSES:
        carpeta = directorio / clase

        if not carpeta.is_dir():
            return False
        if not any(p.suffix.lower() in IMAGE_EXTS for p in carpeta.iterdir()):
            return False

    return True


def extraer_clases(ruta_zip, destino):
    """Descomprime del zip solo las imágenes de las clases seleccionadas."""
    destino = Path(destino)
    extraidas = 0

    with zipfile.ZipFile(ruta_zip) as zf:
        for info in zf.infolist():
            if info.is_dir():
                continue

            partes = Path(info.filename).parts

            # El zip de Kaggle trae el contenido duplicado en dos niveles
            # ('PlantVillage/...' y 'plantvillage/PlantVillage/...'), así que
            # se identifica la clase por el nombre de la carpeta contenedora
            # y se ignora el resto de la ruta.
            if len(partes) < 2 or partes[-2] not in SELECTED_CLASSES:
                continue
            if Path(info.filename).suffix.lower() not in IMAGE_EXTS:
                continue

            salida = destino / partes[-2] / partes[-1]

            if salida.exists():      # evita reescribir el duplicado del zip
                continue

            salida.parent.mkdir(parents=True, exist_ok=True)
            salida.write_bytes(zf.read(info))
            extraidas += 1

    print(f"Imágenes extraídas: {extraidas}")
    return destino

In [ ]:
def find_project_root(start=None):
    """Sube directorios hasta encontrar la carpeta que contiene 'data'."""
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate

    return current


PROJECT_ROOT = find_project_root()

# --- 1. Carpetas ya descomprimidas ----------------------------------------
carpetas_candidatas = [
    Path("/content/plantvillage/PlantVillage"),
    Path("/content/plantvillage"),
    PROJECT_ROOT / "data" / "PlantVillage",
    PROJECT_ROOT / "data",
]

if DRIVE_ROOT is not None:
    carpetas_candidatas = [
        DRIVE_ROOT / "PlantVillage" / "PlantVillage",
        DRIVE_ROOT / "PlantVillage",
    ] + carpetas_candidatas

# Cualquier carpeta subida o descomprimida a mano directo en la sesión de
# Colab (panel de archivos): no se puede predecir su nombre, así que se
# revisa cada subcarpeta de /content (y un nivel más adentro, por si el zip
# de Kaggle trae todo anidado dos veces).
if IN_COLAB and Path("/content").is_dir():
    for p in sorted(Path("/content").iterdir()):
        if p.is_dir() and p.name != "drive":
            carpetas_candidatas += [p, p / "PlantVillage"]

DATA_ROOT = next((c for c in carpetas_candidatas if es_raiz_valida(c)), None)

# --- 2. Si no hay carpeta, buscar un zip y extraer las cinco clases -------
if DATA_ROOT is None:
    zips_candidatos = [
        PROJECT_ROOT / "data" / "raw" / "PlantVillage Dataset.zip",
        PROJECT_ROOT / "data" / "raw" / "PlantVillage.zip",
        PROJECT_ROOT / "PlantVillage.zip",
    ]

    if DRIVE_ROOT is not None:
        zips_candidatos = [
            DRIVE_ROOT / "PlantVillage.zip",
            DRIVE_ROOT / "PlantVillage Dataset.zip",
        ] + zips_candidatos

    # Cualquier .zip subido directo a la sesión de Colab (panel de archivos,
    # ícono de carpeta): tampoco se puede predecir el nombre, así que se
    # revisan todos los .zip sueltos en /content.
    if IN_COLAB and Path("/content").is_dir():
        zips_candidatos += sorted(Path("/content").glob("*.zip"))

    zip_encontrado = next((z for z in zips_candidatos if z.is_file()), None)

    assert zip_encontrado is not None, (
        "No se encontró el dataset.\n"
        "En Colab: sube PlantVillage.zip a la sesión (panel de archivos) o a "
        "'Mi unidad/PlantVillage_MLP/' en Drive.\n"
        "En local: deja el zip en data/raw/ o la carpeta en data/PlantVillage/.\n"
        f"Rutas revisadas: {[str(z) for z in zips_candidatos]}"
    )

    print("Descomprimiendo desde:", zip_encontrado)
    # El zip está en Drive si su ruta cae bajo /content/drive; si no, se subió
    # directo a la sesión (panel de archivos) y no sobrevive un reinicio.
    if IN_COLAB and "/content/drive/" not in str(zip_encontrado):
        print(
            "Nota: este archivo está en la sesión de Colab, no en Drive.\n"
            "Funciona, pero se borra si la sesión se desconecta o reinicia: "
            "tendrías que volver a subirlo. Para que quede guardado entre "
            "sesiones, súbelo a Drive en 'Mi unidad/PlantVillage_MLP/' en su lugar."
        )
    destino = (
        Path("/content/plantvillage") if IN_COLAB
        else PROJECT_ROOT / "data" / "PlantVillage"
    )
    DATA_ROOT = extraer_clases(zip_encontrado, destino)

assert es_raiz_valida(DATA_ROOT), (
    f"La carpeta {DATA_ROOT} no contiene las cinco clases esperadas."
)

# --- 3. Carpetas de salida ------------------------------------------------
# En Colab se escriben en Drive para que no se pierdan al cerrar la sesión.
SALIDA_BASE = DRIVE_ROOT if DRIVE_ROOT is not None else PROJECT_ROOT
IMAGES_DIR = SALIDA_BASE / "images"
MODELS_DIR = SALIDA_BASE / "models"

IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("\nDataset   :", DATA_ROOT)
print("Figuras   :", IMAGES_DIR)
print("Modelos   :", MODELS_DIR)


def save_fig(nombre):
    """Guarda la figura actual para reutilizarla en el informe."""
    destino = IMAGES_DIR / nombre
    plt.savefig(destino, dpi=150, bbox_inches="tight")
    print("Figura guardada:", destino.name)

## 1. Comprensión del problema y EDA

Se mantiene la misma variante de la EP1 para que la comparación MLP-CNN sea objetiva. Las clases, el muestreo, la semilla y la partición no cambian; la diferencia experimental principal será la arquitectura.

In [ ]:
def list_images(directory):
    return sorted(
        path for path in Path(directory).iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    )


# Inventario completo de lo que hay disponible en disco
inventario = pd.Series({
    carpeta.name: len(list_images(carpeta))
    for carpeta in sorted(DATA_ROOT.iterdir())
    if carpeta.is_dir()
}, name="imagenes").sort_values(ascending=False)

print("Clases disponibles:", len(inventario))
print("Imágenes disponibles:", int(inventario.sum()))
inventario.to_frame()

### 2.2 Selección de la variante

Se aplican aquí las tres decisiones que definen la variante del grupo: qué clases,
cuántas imágenes por clase y con qué semilla se muestrean.

In [ ]:
# SELECTED_CLASSES se declaró en la sección 0 porque la extracción del zip
# la necesita. Se muestra aquí para dejar la variante a la vista:
print("Clases de la variante:")
for c in SELECTED_CLASSES:
    print("  -", c)

MAX_IMAGES_PER_CLASS = 400   # variante del grupo: subconjunto acotado

# Etiqueta entera por clase: el orden de SELECTED_CLASSES define el índice,
# y ese mismo orden se reutiliza en la matriz de confusión.
LABEL_TO_ID = {clase: i for i, clase in enumerate(SELECTED_CLASSES)}
ID_TO_LABEL = {i: clase for clase, i in LABEL_TO_ID.items()}

partes = []

for clase in SELECTED_CLASSES:
    rutas = list_images(DATA_ROOT / clase)
    disponibles = pd.DataFrame({"path": [str(p) for p in rutas]})
    disponibles["label"] = clase

    # Muestreo aleatorio reproducible; si una clase tuviera menos de 400
    # imágenes se toman todas las que haya.
    n = min(MAX_IMAGES_PER_CLASS, len(disponibles))
    partes.append(disponibles.sample(n=n, random_state=SEED))

    print(f"{clase:<26} disponibles: {len(rutas):>5}   seleccionadas: {n:>4}")

df = pd.concat(partes, ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)  # mezclar
df["label_id"] = df["label"].map(LABEL_TO_ID).astype("int32")

print("\nTotal de imágenes en el estudio:", len(df))
df.head()

### 2.3 Distribución de clases

Antes de modelar hay que saber si las clases están balanceadas: de eso depende si
el accuracy es una métrica confiable o si hay que apoyarse en F1 y en el recall
por clase.

In [ ]:
conteo = df["label"].value_counts().reindex(SELECTED_CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.barplot(
    x=conteo.values,
    y=[c.replace("Tomato_", "") for c in conteo.index],
    ax=axes[0],
    palette="viridis",
)
axes[0].set_title("Imágenes por clase (subconjunto del estudio)")
axes[0].set_xlabel("cantidad de imágenes")

axes[1].pie(
    conteo.values,
    labels=[c.replace("Tomato_", "") for c in conteo.index],
    autopct="%1.1f%%",
    startangle=90,
)
axes[1].set_title("Proporción de cada clase")

plt.tight_layout()
save_fig("02_distribucion_clases.png")
plt.show()

desbalance = conteo.max() / conteo.min()
print(f"Razón de desbalance (máx/mín): {desbalance:.2f}")
print("Conjunto balanceado" if desbalance < 1.5 else "Conjunto desbalanceado")

**Lectura:** el muestreo tope de 400 por clase dejó las cinco clases con el mismo
número de imágenes, así que el conjunto quedó perfectamente balanceado. Dos
consecuencias prácticas:

1. El **accuracy es interpretable**: no puede inflarse prediciendo siempre la
   clase mayoritaria, porque no hay clase mayoritaria.
2. La **línea base al azar es 20 %** (1/5). Cualquier resultado cercano a ese
   valor significa que la red no aprendió nada.

Aun así se reportan Precision, Recall y F1 por clase, porque el accuracy global
puede esconder que una clase específica se está clasificando mal.

### 2.4 Control de calidad

Se revisan las imágenes seleccionadas para descartar archivos corruptos,
resoluciones inconsistentes o imágenes en escala de grises que romperían el
supuesto de 3 canales.

In [ ]:
registros = []

for fila in df.itertuples():
    try:
        with Image.open(fila.path) as img:
            registros.append({
                "label": fila.label,
                "ancho": img.width,
                "alto": img.height,
                "modo": img.mode,
                "corrupta": False,
            })
    except Exception:
        registros.append({
            "label": fila.label,
            "ancho": np.nan,
            "alto": np.nan,
            "modo": None,
            "corrupta": True,
        })

calidad = pd.DataFrame(registros)

print("Imágenes corruptas:", int(calidad["corrupta"].sum()))
print("Modos de color:", calidad["modo"].value_counts().to_dict())
print("Resoluciones distintas:",
      calidad.groupby(["ancho", "alto"]).size().to_dict())
print("\nValores duplicados de ruta:", int(df["path"].duplicated().sum()))
calidad.describe()

**Lectura:** no hay archivos corruptos, todas las imágenes son RGB y comparten la
resolución 256 × 256. El conjunto está limpio, así que **no se requiere imputación
ni descarte de registros**. Esto no es casualidad: PlantVillage se construyó en
laboratorio con protocolo fijo, y esa uniformidad es precisamente lo que después
limitará la generalización del modelo a fotos reales de campo.

### 2.5 Exploración visual

Ver las imágenes es parte del análisis, no decoración: permite anticipar qué
clases van a confundirse antes de entrenar.

In [ ]:
fig, axes = plt.subplots(
    len(SELECTED_CLASSES), 5,
    figsize=(14, 3 * len(SELECTED_CLASSES)),
)

for i, clase in enumerate(SELECTED_CLASSES):
    muestras = df[df["label"] == clase].head(5)

    for j, fila in enumerate(muestras.itertuples()):
        with Image.open(fila.path) as img:
            axes[i, j].imshow(img)
        axes[i, j].axis("off")

        if j == 0:
            axes[i, j].set_title(
                clase.replace("Tomato_", ""),
                loc="left", fontsize=11, fontweight="bold",
            )

plt.suptitle("Cinco ejemplos por clase", y=1.0, fontsize=13)
plt.tight_layout()
save_fig("02_ejemplos_por_clase.png")
plt.show()

### 2.6 Color promedio por clase

Un MLP sobre píxeles planos no detecta formas: lo que puede aprovechar son
diferencias globales de color e intensidad. Conviene medir cuánta señal hay ahí.

In [ ]:
def color_promedio(ruta, tamano=(64, 64)):
    with Image.open(ruta) as img:
        arr = np.asarray(img.convert("RGB").resize(tamano), dtype=np.float32)
    return arr.reshape(-1, 3).mean(axis=0)


# Se usa una submuestra por clase: suficiente para estimar la tendencia
muestra = df.groupby("label", group_keys=False).head(80)
canales = np.stack([color_promedio(p) for p in muestra["path"]])

color_df = pd.DataFrame(canales, columns=["R", "G", "B"])
color_df["label"] = muestra["label"].values
color_df["brillo"] = color_df[["R", "G", "B"]].mean(axis=1)

resumen_color = (
    color_df.groupby("label")[["R", "G", "B", "brillo"]]
    .mean()
    .reindex(SELECTED_CLASSES)
    .round(1)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

resumen_color[["R", "G", "B"]].plot(
    kind="bar", ax=axes[0], color=["#c0392b", "#27ae60", "#2980b9"],
)
axes[0].set_title("Intensidad media por canal RGB")
axes[0].set_ylabel("valor medio (0-255)")
axes[0].set_xticklabels(
    [c.replace("Tomato_", "") for c in resumen_color.index],
    rotation=30, ha="right",
)

sns.boxplot(
    data=color_df, x="label", y="brillo", ax=axes[1], palette="viridis",
)
axes[1].set_title("Distribución del brillo por clase")
axes[1].set_xticklabels(
    [c.replace("Tomato_", "") for c in SELECTED_CLASSES],
    rotation=30, ha="right",
)

plt.tight_layout()
save_fig("02_color_por_clase.png")
plt.show()

resumen_color

**Lectura:** las clases **no** se separan de forma homogénea por color global.

- `Bacterial_spot` y `Leaf_Mold` sí tienen una firma de color propia: la primera
  es notablemente más oscura que el resto, la segunda destaca por su canal azul.
- `Early_blight` y `Late_blight` tienen el perfil de color **más parecido entre
  sí de todo el conjunto** (sus medias de brillo son casi idénticas y los
  boxplots se solapan casi por completo). Ambas son enfermedades fúngicas que
  producen lesiones café sobre tejido verde; lo que las diferencia es la *forma*
  de la lesión —anillos concéntricos frente a manchas irregulares—, no el color.
- `healthy` es un caso aparte: no tiene lesión que darle una firma de color
  particular, así que su color medio queda en un punto intermedio del grupo. Su
  rasgo distintivo no es cromático sino la **ausencia de cualquier textura
  anómala** en toda la hoja — una propiedad más difícil de anticipar solo con
  este gráfico.

**Predicción verificable:** si un MLP sobre píxeles planos solo puede explotar
estadísticos globales de color, el par `Early_blight`/`Late_blight` —el más
parecido en color— debería ser el que más se confunde entre sí, mientras que
`Bacterial_spot` y `Leaf_Mold` deberían clasificarse mejor por tener firma
propia. Qué pasa con `healthy` queda abierto: depende de si el modelo logra
capturar la ausencia de textura como patrón, algo que ni este gráfico ni el
razonamiento de esta sección permiten anticipar. Esta predicción se contrasta
contra la matriz de confusión real en la **sección 7.5**.

La razón de fondo por la que el color pesa tanto es que `Flatten()` destruye la
vecindad entre píxeles, y con ella la noción de "anillo concéntrico". La red
queda con 12.288 valores sin estructura espacial, y lo único que le resulta
fácil de explotar es un promedio global como el color.

## 2. Preparación de imágenes para CNN

La CNN recibe un tensor de tres dimensiones por imagen: alto, ancho y canales. A diferencia del MLP, no se aplica `Flatten` durante el preprocesamiento. La capa convolucional será la encargada de aprender representaciones espaciales.

## 3. Preprocesamiento

Cuatro transformaciones, cada una con su justificación técnica.

### 3.1 Partición estratificada

Se separa **antes** de cualquier ajuste, para que ninguna información del conjunto
de prueba influya en el entrenamiento (evitar *data leakage*).

- **70 % entrenamiento** — ajustar los pesos.
- **15 % validación** — decidir cuándo detener y comparar configuraciones.
- **15 % prueba** — se toca **una sola vez**, al final, para estimar generalización.

`stratify` mantiene la proporción de las cinco clases en cada subconjunto: sin
eso, el azar podría dejar una clase sub-representada en prueba y las métricas de
esa clase serían ruido.

In [ ]:
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15

# Primer corte: separa entrenamiento del resto (validación + prueba)
train_df, temp_df = train_test_split(
    df,
    train_size=TRAIN_SIZE,
    stratify=df["label_id"],
    random_state=SEED,
)

# Segundo corte: divide el resto en mitades iguales → 15 % y 15 %
val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE),
    stratify=temp_df["label_id"],
    random_state=SEED,
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

reparto = pd.DataFrame({
    "entrenamiento": train_df["label"].value_counts(),
    "validación": val_df["label"].value_counts(),
    "prueba": test_df["label"].value_counts(),
}).reindex(SELECTED_CLASSES)
reparto.loc["TOTAL"] = reparto.sum()

print(f"Entrenamiento: {len(train_df)} imágenes")
print(f"Validación   : {len(val_df)} imágenes")
print(f"Prueba       : {len(test_df)} imágenes")

# Verificación explícita: ninguna imagen puede estar en dos subconjuntos
solapamiento = (
    set(train_df["path"]) & set(val_df["path"])
    | set(train_df["path"]) & set(test_df["path"])
    | set(val_df["path"]) & set(test_df["path"])
)
assert not solapamiento, "Hay imágenes repetidas entre subconjuntos"
print("\nSin solapamiento entre subconjuntos ✓")

reparto

### 3.2 Carga, redimensionado y normalización

Tres pasos sobre cada imagen:

1. **Redimensionar a 64 × 64.** Un MLP conecta cada píxel de entrada con cada
   neurona de la primera capa. A 256 × 256 × 3 serían 196.608 entradas y, con 256
   neuronas, más de 50 millones de parámetros solo en la primera capa: imposible
   de entrenar con 1.400 imágenes sin memorizar. A 64 × 64 × 3 quedan 12.288
   entradas — se pierde detalle fino, pero se conserva la mancha y el modelo
   vuelve a ser entrenable. **Es una decisión de compromiso, no un valor arbitrario.**

2. **Normalizar a [0, 1]** dividiendo por 255. Con valores 0–255 las sumas
   ponderadas iniciales son enormes, los gradientes se desestabilizan y el
   entrenamiento puede no converger. Es la *regla de oro* vista en clase:
   siempre escalar antes de entrenar.

3. **Aplanar con `Flatten()`.** La matriz 64 × 64 × 3 se convierte en un vector de
   12.288 valores, porque una capa `Dense` solo acepta vectores. **Aquí se pierde
   la estructura espacial** — el costo conceptual central de usar un MLP en
   imágenes, y el que se analiza en la sección 7.

In [ ]:
IMG_SIZE = (64, 64)
N_CHANNELS = 3
INPUT_DIM = IMG_SIZE[0] * IMG_SIZE[1] * N_CHANNELS
NUM_CLASSES = len(SELECTED_CLASSES)

print(f"Entrada por imagen: {IMG_SIZE[0]}×{IMG_SIZE[1]}×{N_CHANNELS} = {INPUT_DIM} valores")
print(f"Clases de salida  : {NUM_CLASSES}")


def cargar_conjunto(dataframe, nombre):
    """Carga un subconjunto completo a memoria como arreglos de NumPy.

    Con 2.000 imágenes de 64×64 el conjunto entero ocupa ~100 MB, así que
    cabe en memoria y se evita la complejidad de un pipeline por lotes.
    """
    X = np.zeros((len(dataframe), *IMG_SIZE, N_CHANNELS), dtype=np.float32)

    for i, ruta in enumerate(dataframe["path"]):
        with Image.open(ruta) as img:
            arr = np.asarray(
                img.convert("RGB").resize(IMG_SIZE, Image.BILINEAR),
                dtype=np.float32,
            )
        X[i] = arr / 255.0          # normalización a [0, 1]

    y = dataframe["label_id"].to_numpy(dtype=np.int32)
    print(f"{nombre:<14} X={X.shape}  y={y.shape}")
    return X, y


X_train, y_train = cargar_conjunto(train_df, "entrenamiento")
X_val, y_val = cargar_conjunto(val_df, "validación")
X_test, y_test = cargar_conjunto(test_df, "prueba")

In [ ]:
# Verificación del preprocesamiento: si esto falla, el modelo no aprenderá bien
print(f"Rango de píxeles en entrenamiento: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Media: {X_train.mean():.3f}   Desviación estándar: {X_train.std():.3f}")

assert 0.0 <= X_train.min() and X_train.max() <= 1.0, \
    "Los píxeles no están normalizados en [0, 1]"
assert not np.isnan(X_train).any(), "Hay valores NaN en las imágenes"
print("\nNormalización verificada ✓")

# Comparación visual: original vs. la entrada real que recibe la red
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for j in range(5):
    with Image.open(train_df.loc[j, "path"]) as img:
        axes[0, j].imshow(img)
    axes[0, j].set_title(
        train_df.loc[j, "label"].replace("Tomato_", ""), fontsize=9,
    )
    axes[0, j].axis("off")

    axes[1, j].imshow(X_train[j])
    axes[1, j].set_title("64×64 normalizada", fontsize=9)
    axes[1, j].axis("off")

axes[0, 0].set_ylabel("original")
plt.suptitle("Arriba: original 256×256  |  Abajo: entrada de la red 64×64", fontsize=12)
plt.tight_layout()
save_fig("03_preprocesamiento.png")
plt.show()

**Lectura:** en las imágenes de 64 × 64 las lesiones siguen siendo visibles, pero
el detalle fino de los bordes se degrada. Ese detalle es parte de lo que
diferencia `Early_blight` de `Late_blight`, así que la reducción de resolución
—necesaria para que el MLP sea entrenable— ya introduce una pérdida de
información que se va a reflejar en la matriz de confusión.

## 3. Diseño de la CNN

La arquitectura utiliza tres bloques convolucionales. Cada bloque combina una convolución, Batch Normalization y MaxPooling. La reducción espacial permite conservar patrones importantes y disminuir el costo computacional. Al final, Global Average Pooling transforma los mapas de características en un vector compacto antes de las capas densas.

### Arquitectura propuesta

| Etapa | Componentes | Propósito |
|---|---|---|
| Bloque 1 | Conv2D 32 filtros de 3×3 + MaxPooling | Detectar bordes y texturas simples |
| Bloque 2 | Conv2D 64 filtros de 3×3 + MaxPooling | Combinar texturas y patrones intermedios |
| Bloque 3 | Conv2D 128 filtros de 3×3 + MaxPooling | Representar lesiones y estructuras más complejas |
| Clasificación | GlobalAveragePooling + Dense 128 + Dropout + Softmax | Clasificar las cinco clases |

In [ ]:
def construir_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES):
    modelo = Sequential(name="CNN_PlantVillage")

    modelo.add(Input(shape=input_shape, name="entrada"))
    modelo.add(tf.keras.layers.RandomFlip("horizontal", name="aumento_flip"))
    modelo.add(tf.keras.layers.RandomRotation(0.05, name="aumento_rotacion"))

    modelo.add(Conv2D(32, (3, 3), padding="same", name="conv_1"))
    modelo.add(BatchNormalization(name="bn_1"))
    modelo.add(Activation("relu", name="relu_1"))
    modelo.add(MaxPooling2D((2, 2), name="pool_1"))

    modelo.add(Conv2D(64, (3, 3), padding="same", name="conv_2"))
    modelo.add(BatchNormalization(name="bn_2"))
    modelo.add(Activation("relu", name="relu_2"))
    modelo.add(MaxPooling2D((2, 2), name="pool_2"))

    modelo.add(Conv2D(128, (3, 3), padding="same", name="conv_3"))
    modelo.add(BatchNormalization(name="bn_3"))
    modelo.add(Activation("relu", name="relu_3"))
    modelo.add(MaxPooling2D((2, 2), name="pool_3"))

    modelo.add(GlobalAveragePooling2D(name="global_average_pooling"))
    modelo.add(Dense(128, activation="relu", name="dense_1"))
    modelo.add(Dropout(0.40, name="dropout"))
    modelo.add(Dense(num_classes, activation="softmax", name="salida"))

    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    return modelo


modelo_cnn = construir_cnn()
modelo_cnn.summary()
print("Parámetros CNN:", modelo_cnn.count_params())

## 4. Entrenamiento de la CNN

Se utilizan los mismos conjuntos de entrenamiento y validación definidos anteriormente. Early Stopping conserva los mejores pesos según `val_loss`; ReduceLROnPlateau disminuye la tasa de aprendizaje cuando la validación deja de mejorar.

In [ ]:
EPOCHS_CNN = 60
BATCH_SIZE_CNN = 32

callbacks_cnn = [
    EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
    ),
]

inicio_cnn = time.perf_counter()

history_cnn = modelo_cnn.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_CNN,
    batch_size=BATCH_SIZE_CNN,
    callbacks=callbacks_cnn,
    verbose=1,
)

tiempo_cnn = time.perf_counter() - inicio_cnn
print(f"Tiempo de entrenamiento CNN: {tiempo_cnn:.2f} segundos")

In [ ]:
hist_cnn = pd.DataFrame(history_cnn.history)
mejor_epoca_cnn = int(hist_cnn["val_loss"].idxmin() + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(hist_cnn["loss"], label="Entrenamiento")
axes[0].plot(hist_cnn["val_loss"], label="Validación")
axes[0].set_title("CNN: pérdida")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(hist_cnn["accuracy"], label="Entrenamiento")
axes[1].plot(hist_cnn["val_accuracy"], label="Validación")
axes[1].set_title("CNN: accuracy")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
save_fig("08_cnn_curvas_aprendizaje.png")
plt.show()

print("Mejor época CNN:", mejor_epoca_cnn)

## 5. Evaluación de la CNN

In [ ]:
def calcular_metricas(modelo, X, y, nombre):
    probabilidades = modelo.predict(X, verbose=0)
    predicciones = probabilidades.argmax(axis=1)

    resultado = {
        "modelo": nombre,
        "accuracy": accuracy_score(y, predicciones),
        "precision_macro": precision_score(y, predicciones, average="macro", zero_division=0),
        "recall_macro": recall_score(y, predicciones, average="macro", zero_division=0),
        "f1_macro": f1_score(y, predicciones, average="macro", zero_division=0),
    }

    return resultado, probabilidades, predicciones


metricas_cnn, prob_cnn, pred_cnn = calcular_metricas(
    modelo_cnn,
    X_test,
    y_test,
    "CNN",
)

metricas_cnn_df = pd.DataFrame([metricas_cnn])
display(metricas_cnn_df.round(4))

reporte_cnn = pd.DataFrame(
    classification_report(
        y_test,
        pred_cnn,
        labels=list(range(NUM_CLASSES)),
        target_names=SELECTED_CLASSES,
        output_dict=True,
        zero_division=0,
    )
).T

display(reporte_cnn.round(4))

In [ ]:
cm_cnn = confusion_matrix(y_test, pred_cnn, labels=list(range(NUM_CLASSES)))

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_cnn,
    annot=True,
    fmt="d",
    cmap="Greens",
    xticklabels=SELECTED_CLASSES,
    yticklabels=SELECTED_CLASSES,
)
plt.title("CNN: matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Valor real")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
save_fig("08_cnn_matriz_confusion.png")
plt.show()

## 6. Comparación experimental MLP vs CNN

Para que la comparación sea objetiva, se reentrena una versión equivalente del MLP usando exactamente los mismos `X_train`, `X_val`, `X_test`, etiquetas, semilla y callbacks. También se conserva como referencia el resultado reportado en la EP1.

In [ ]:
def construir_mlp_comparacion():
    modelo = Sequential(name="MLP_comparacion_EP2")
    modelo.add(Input(shape=(*IMG_SIZE, 3), name="entrada"))
    modelo.add(tf.keras.layers.RandomFlip("horizontal"))
    modelo.add(tf.keras.layers.RandomRotation(0.05))
    modelo.add(Flatten())

    for unidades in (256, 128):
        modelo.add(Dense(unidades, kernel_initializer="he_normal"))
        modelo.add(BatchNormalization())
        modelo.add(Activation("relu"))
        modelo.add(Dropout(0.30))

    modelo.add(Dense(NUM_CLASSES, activation="softmax", name="salida"))
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return modelo


modelo_mlp = construir_mlp_comparacion()

inicio_mlp = time.perf_counter()
history_mlp = modelo_mlp.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS_CNN,
    batch_size=BATCH_SIZE_CNN,
    callbacks=[
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ],
    verbose=1,
)
tiempo_mlp = time.perf_counter() - inicio_mlp

metricas_mlp, prob_mlp, pred_mlp = calcular_metricas(
    modelo_mlp,
    X_test,
    y_test,
    "MLP reentrenado",
)

hist_mlp = pd.DataFrame(history_mlp.history)
mejor_epoca_mlp = int(hist_mlp["val_loss"].idxmin() + 1)

comparacion = pd.DataFrame([
    {
        **metricas_mlp,
        "parametros": modelo_mlp.count_params(),
        "tiempo_segundos": tiempo_mlp,
        "mejor_epoca": mejor_epoca_mlp,
    },
    {
        **metricas_cnn,
        "parametros": modelo_cnn.count_params(),
        "tiempo_segundos": tiempo_cnn,
        "mejor_epoca": mejor_epoca_cnn,
    },
])

comparacion["accuracy_train"] = [
    accuracy_score(y_train, modelo_mlp.predict(X_train, verbose=0).argmax(axis=1)),
    accuracy_score(y_train, modelo_cnn.predict(X_train, verbose=0).argmax(axis=1)),
]
comparacion["brecha_generalizacion"] = comparacion["accuracy_train"] - comparacion["accuracy"]

display(comparacion.round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

comparacion.plot(x="modelo", y="accuracy", kind="bar", ax=axes[0], color=["#7f8c8d", "#27ae60"], legend=False)
axes[0].set_title("Accuracy en prueba")
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Accuracy")

comparacion.plot(x="modelo", y="f1_macro", kind="bar", ax=axes[1], color=["#7f8c8d", "#27ae60"], legend=False)
axes[1].set_title("F1 macro en prueba")
axes[1].set_ylim(0, 1)
axes[1].set_ylabel("F1 macro")

comparacion.plot(x="modelo", y="parametros", kind="bar", ax=axes[2], color=["#7f8c8d", "#27ae60"], legend=False)
axes[2].set_title("Cantidad de parámetros")
axes[2].set_ylabel("Parámetros")

plt.tight_layout()
save_fig("09_comparacion_mlp_cnn.png")
plt.show()

### Interpretación de la comparación

La CNN debería aprovechar la estructura espacial de las hojas, por lo que se espera que supere o iguale al MLP en F1 macro y en capacidad de generalización. La conclusión debe basarse en la tabla generada: no se debe afirmar que la CNN es mejor solo por tener convoluciones.

- Si la CNN obtiene mayor F1 y menor brecha, generaliza mejor.
- Si el MLP obtiene un resultado similar con menos tiempo o parámetros, se debe reportar como una ventaja de eficiencia del MLP.
- La CNN ofrece una representación interpretable mediante Feature Maps y mapas de activación; el MLP no tiene ese mecanismo espacial directo.

## 7. Feature Maps y mapas de activación

Los Feature Maps muestran qué patrones responde cada filtro de las capas convolucionales. En capas iniciales suelen aparecer bordes y texturas; en capas profundas las activaciones representan combinaciones más complejas asociadas a lesiones o regiones de la hoja.

In [ ]:
indice_muestra = 0
imagen_muestra = X_test[indice_muestra]
etiqueta_muestra = SELECTED_CLASSES[y_test[indice_muestra]]

capas_conv = [
    capa for capa in modelo_cnn.layers
    if isinstance(capa, Conv2D)
]

extractor_feature_maps = tf.keras.Model(
    inputs=modelo_cnn.input,
    outputs=[capa.output for capa in capas_conv],
)

activaciones = extractor_feature_maps.predict(
    imagen_muestra[None, ...],
    verbose=0,
)

fig, axes = plt.subplots(
    len(activaciones),
    8,
    figsize=(16, 3 * len(activaciones)),
)

axes = np.atleast_2d(axes)

for fila, (activacion, capa) in enumerate(zip(activaciones, capas_conv)):
    mapas = activacion[0]
    for columna in range(8):
        axes[fila, columna].imshow(mapas[:, :, columna], cmap="viridis")
        axes[fila, columna].axis("off")
        if columna == 0:
            axes[fila, columna].set_ylabel(capa.name, fontsize=10)

plt.suptitle(f"Feature Maps - clase real: {etiqueta_muestra}")
plt.tight_layout()
save_fig("10_feature_maps.png")
plt.show()

In [ ]:
ultima_capa_conv = modelo_cnn.get_layer("conv_3")
modelo_gradcam = tf.keras.models.Model(
    inputs=modelo_cnn.inputs,
    outputs=[ultima_capa_conv.output, modelo_cnn.output],
)


def generar_gradcam(imagen, class_index):
    tensor = tf.convert_to_tensor(imagen[None, ...])

    with tf.GradientTape() as tape:
        conv_outputs, predicciones = modelo_gradcam(tensor, training=False)
        class_score = predicciones[:, class_index]

    gradientes = tape.gradient(class_score, conv_outputs)
    pesos = tf.reduce_mean(gradientes, axis=(1, 2))
    conv_outputs = conv_outputs[0]
    pesos = pesos[0]

    heatmap = tf.reduce_sum(conv_outputs * pesos[None, None, :], axis=-1)
    heatmap = tf.maximum(heatmap, 0)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()


indices_por_clase = {
    class_id: int(np.where(y_test == class_id)[0][0])
    for class_id in range(NUM_CLASSES)
}

fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(10, 3 * NUM_CLASSES))

for class_id, indice in indices_por_clase.items():
    heatmap = generar_gradcam(X_test[indice], class_id)
    prediccion = int(pred_cnn[indice])

    axes[class_id, 0].imshow(X_test[indice])
    axes[class_id, 0].set_title(
        f"Real: {SELECTED_CLASSES[class_id]}\nPred: {SELECTED_CLASSES[prediccion]}"
    )
    axes[class_id, 0].axis("off")

    axes[class_id, 1].imshow(X_test[indice])
    axes[class_id, 1].imshow(
        heatmap,
        cmap="jet",
        alpha=0.45,
        extent=(0, IMG_SIZE[1], IMG_SIZE[0], 0),
    )
    axes[class_id, 1].set_title(f"Activación para {SELECTED_CLASSES[class_id]}")
    axes[class_id, 1].axis("off")

plt.tight_layout()
save_fig("11_mapas_activacion_por_clase.png")
plt.show()

## 8. Análisis de errores de la CNN

In [ ]:
resultados_cnn = test_df.reset_index(drop=True).copy()
resultados_cnn["real"] = [SELECTED_CLASSES[i] for i in y_test]
resultados_cnn["prediccion"] = [SELECTED_CLASSES[i] for i in pred_cnn]
resultados_cnn["confianza"] = prob_cnn.max(axis=1)

errores_cnn = resultados_cnn[resultados_cnn["real"] != resultados_cnn["prediccion"]]
aciertos_cnn = resultados_cnn[resultados_cnn["real"] == resultados_cnn["prediccion"]]

print("Aciertos CNN:", len(aciertos_cnn))
print("Errores CNN:", len(errores_cnn))
print("Tasa de error CNN:", round(len(errores_cnn) / len(resultados_cnn), 4))

display(
    errores_cnn[["real", "prediccion", "confianza", "path"]]
    .sort_values("confianza", ascending=False)
    .head(10)
)

In [ ]:
def mostrar_ejemplos_cnn(dataframe, titulo, n=10):
    subconjunto = dataframe.head(n)
    if len(subconjunto) == 0:
        print("No hay ejemplos para mostrar")
        return

    columnas = min(5, len(subconjunto))
    filas = int(np.ceil(len(subconjunto) / columnas))
    fig, axes = plt.subplots(filas, columnas, figsize=(18, 4 * filas))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for ax, (_, row) in zip(axes, subconjunto.iterrows()):
        with Image.open(row["path"]) as img:
            ax.imshow(img.convert("RGB"))
        ax.set_title(
            f"Real: {row['real'].replace('Tomato_', '')}\n"
            f"Pred: {row['prediccion'].replace('Tomato_', '')}\n"
            f"Confianza: {row['confianza']:.2f}",
            fontsize=9,
        )
        ax.axis("off")

    plt.suptitle(titulo)
    plt.tight_layout()
    plt.show()


mostrar_ejemplos_cnn(
    errores_cnn.sort_values("confianza", ascending=False),
    "CNN: errores de alta confianza",
)

mostrar_ejemplos_cnn(
    aciertos_cnn.sort_values("confianza", ascending=False),
    "CNN: ejemplos correctamente clasificados",
)

## 9. Conclusiones de la EP2

Completar después de ejecutar el notebook:

- La CNN obtuvo un Accuracy de **[completar]** y un F1 macro de **[completar]**.
- El MLP reentrenado obtuvo un Accuracy de **[completar]** y un F1 macro de **[completar]**.
- La arquitectura con mejor generalización fue **[completar]**, considerando la brecha entre Accuracy de entrenamiento y prueba.
- Los Feature Maps iniciales muestran **[completar]**; los mapas profundos se concentran en **[completar]**.
- La principal ventaja de la CNN es conservar relaciones espaciales; su principal costo es el mayor tiempo y complejidad de entrenamiento.

## 10. Guardar artefactos

In [ ]:
modelo_cnn.save(str(MODELS_DIR / "cnn_plantvillage.keras"))

hist_cnn.to_csv(MODELS_DIR / "historial_cnn.csv", index=False)
comparacion.to_csv(MODELS_DIR / "comparacion_mlp_cnn.csv", index=False)
metricas_cnn_df.to_csv(MODELS_DIR / "metricas_cnn.csv", index=False)
pd.DataFrame(cm_cnn, index=SELECTED_CLASSES, columns=SELECTED_CLASSES).to_csv(
    MODELS_DIR / "matriz_confusion_cnn.csv"
)

resultados_cnn.to_csv(MODELS_DIR / "predicciones_test_cnn.csv", index=False)

print("Artefactos CNN guardados en:", MODELS_DIR)